# qlib recorder regime/window 分析入口验证

目标：验证 `src.analysis` 可以把 qlib recorder 风格的 portfolio/risk artifact 直接转换成统一 `AnalysisReport`，并输出 regime/window 模型差异、segment effect matrix 和中文 review briefing。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analysis import (
    build_qlib_recorder_regime_window_analysis_report,
    load_qlib_recorder_analysis,
    render_model_review_briefing_markdown,
    write_analysis_report,
)
from src.core import DatasetSnapshot

EXPERIMENT = "qlib_recorder_regime_window_v1"
OUTPUT_DIR = ROOT / "research" / EXPERIMENT
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
class FakeQlibRecorder:
    def __init__(self, objects: dict[str, object]):
        self.objects = objects

    def load_object(self, name: str):
        if name not in self.objects:
            raise FileNotFoundError(name)
        return self.objects[name]


def qlib_report(returns: list[float], benchmark: list[float]) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "return": returns,
            "bench": benchmark,
            "cost": [0.0, 0.001, 0.002, 0.001, 0.002, 0.001, 0.001, 0.001],
        },
        index=pd.date_range("2026-01-01", periods=8, freq="D"),
    )


ridge = load_qlib_recorder_analysis(
    FakeQlibRecorder(
        {
            "portfolio_analysis/report_normal_1day.pkl": qlib_report(
                [0.0, 0.02, 0.02, -0.01, -0.015, -0.01, 0.005, 0.01],
                [0.0, 0.01, 0.01, -0.01, -0.02, -0.01, 0.0, 0.0],
            ),
            "portfolio_analysis/risk_analysis_1day.pkl": {
                "annualized_return": 0.18,
                "information_ratio": 0.90,
                "max_drawdown": -0.04,
            },
        }
    ),
    model_name="ridge_mse",
)
gcn = load_qlib_recorder_analysis(
    FakeQlibRecorder(
        {
            "portfolio_analysis/report_normal_1day.pkl": qlib_report(
                [0.0, 0.04, 0.04, -0.01, -0.06, -0.04, 0.01, 0.01],
                [0.0, 0.01, 0.01, -0.01, -0.02, -0.01, 0.0, 0.0],
            ),
            "portfolio_analysis/risk_analysis_1day.pkl": {
                "annualized_return": 0.22,
                "information_ratio": 0.70,
                "max_drawdown": -0.11,
            },
        }
    ),
    model_name="gcn_mse",
)

regime_frame = pd.DataFrame(
    {
        "date": pd.date_range("2026-01-01", periods=8, freq="D"),
        "regime": ["bull", "bull", "bull", "neutral", "stress", "stress", "stress", "downtrend"],
        "vix": [17, 18, 19, 24, 36, 34, 31, 23],
        "trend": [0.02, 0.03, 0.04, 0.00, -0.06, -0.05, -0.03, -0.02],
        "drawdown": [0.0, 0.0, 0.0, -0.01, -0.12, -0.15, -0.10, -0.04],
    }
)
windows = {
    "early": ("2026-01-01", "2026-01-04"),
    "stress_window": ("2026-01-05", "2026-01-07"),
}

In [3]:
report = build_qlib_recorder_regime_window_analysis_report(
    artifact_id=EXPERIMENT,
    recorder_analyses={"ridge_mse": ridge, "gcn_mse": gcn},
    regime_frame=regime_frame,
    windows=windows,
    baseline_model="ridge_mse",
    summary={
        "note": "中文 review 入口：从 qlib recorder artifact 直接生成 regime/window 模型差异。",
    },
)
dataset = DatasetSnapshot(
    name="qlib_recorder_synthetic",
    universe=("SOXL", "SOXX"),
    start="2026-01-01",
    end="2026-01-08",
    calendar_path="data/qlib/soxl/calendars/day.txt",
    data_paths=("synthetic_qlib_recorder",),
)
export_result = write_analysis_report(report, OUTPUT_DIR, dataset=dataset)
briefing_md = render_model_review_briefing_markdown(
    report.tables["model_review_briefing"],
    title="qlib recorder 模型差异 Review",
)
(OUTPUT_DIR / "model_review_briefing.md").write_text(briefing_md, encoding="utf-8")

print(json.dumps(export_result, ensure_ascii=False, indent=2))

{
  "schema_version": "analysis.report.export.v1",
  "artifact_id": "qlib_recorder_regime_window_v1",
  "artifact_type": "analysis_report",
  "paths": {
    "summary": "analysis_summary.json",
    "model_comparison": "model_comparison.csv",
    "regime_metrics": "regime_metrics.csv",
    "window_metrics": "window_metrics.csv",
    "window_regime_metrics": "window_regime_metrics.csv",
    "qlib_return_frame": "qlib_return_frame.csv",
    "qlib_risk_metrics": "qlib_risk_metrics.csv",
    "segment_metrics": "segment_metrics.csv",
    "segment_effect_matrix": "segment_effect_matrix.csv",
    "model_effect_deltas": "model_effect_deltas.csv",
    "model_review_briefing": "model_review_briefing.csv",
    "failure_modes": "failure_modes.csv",
    "qlib_portfolio_report": "qlib_portfolio_report.csv",
    "qlib_risk_analysis": "qlib_risk_analysis.csv",
    "manifest": "analysis_manifest.json"
  },
  "table_counts": {
    "failure_modes": 4,
    "model_comparison": 2,
    "model_effect_deltas": 2

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [4]:
summary = {
    "schema_version": "research.qlib_recorder_regime_window_v1",
    "status": report.summary["status"],
    "source_model_count": report.summary["source_model_count"],
    "baseline_model": report.summary["analysis_summary"]["baseline_model"],
    "table_counts": report.table_counts(),
    "review_signals": sorted(report.tables["model_review_briefing"]["review_signal"].unique().tolist()),
    "decision_hints": sorted(report.tables["model_review_briefing"]["decision_hint"].unique().tolist()),
    "weakest_segment": report.summary["analysis_summary"]["weakest_segments"][0],
    "export_paths": export_result["paths"],
}
(OUTPUT_DIR / "notebook_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame(report.tables["segment_effect_matrix"]).head(12)

,schema_version,segment_type,segment,window,regime,baseline_model,model_count,best_relative_return_model,best_relative_return_pct,worst_relative_return_model,...,best_stability_vs_baseline_delta,highest_drawdown_model,highest_drawdown_pct,lowest_drawdown_model,lowest_drawdown_pct,drawdown_pct_spread,highest_turnover_model,highest_turnover,average_turnover,review_note
0,analysis.segment_effect_matrix.v1,all,all,,,ridge_mse,2,ridge_mse,3.971282,gcn_mse,...,0.000000,gcn_mse,10.6624,ridge_mse,3.46015,7.202250e+00,gcn_mse,0.0,0.0,ridge_mse 在 all 领先；gcn_mse 明显弱于基准，需检查该分段适配。
1,analysis.segment_effect_matrix.v1,regime,bull,,bull,ridge_mse,2,gcn_mse,6.150000,ridge_mse,...,4.120000,gcn_mse,0.0000,gcn_mse,0.00000,0.000000e+00,gcn_mse,0.0,0.0,gcn_mse 在 bull 强于基准 ridge_mse，可作为该 regime/wind...
2,analysis.segment_effect_matrix.v1,regime,downtrend,,downtrend,ridge_mse,2,gcn_mse,0.000000,gcn_mse,...,0.000000,gcn_mse,0.0000,gcn_mse,0.00000,0.000000e+00,gcn_mse,0.0,0.0,downtrend 模型差异较小，需结合回撤、换手和组合互补性判断。
3,analysis.segment_effect_matrix.v1,regime,neutral,,neutral,ridge_mse,2,gcn_mse,0.000000,gcn_mse,...,0.000000,gcn_mse,0.0000,gcn_mse,0.00000,0.000000e+00,gcn_mse,0.0,0.0,neutral 模型差异较小，需结合回撤、换手和组合互补性判断。
4,analysis.segment_effect_matrix.v1,regime,stress,,stress,ridge_mse,2,ridge_mse,0.495000,gcn_mse,...,0.000000,gcn_mse,4.0000,ridge_mse,1.00000,3.000000e+00,gcn_mse,0.0,0.0,ridge_mse 在 stress 领先；gcn_mse 明显弱于基准，需检查该分段适配。
5,analysis.segment_effect_matrix.v1,window,early,early,,ridge_mse,2,gcn_mse,6.088500,ridge_mse,...,48.978689,ridge_mse,1.0000,gcn_mse,1.00000,2.220446e-14,gcn_mse,0.0,0.0,gcn_mse 在 early 强于基准 ridge_mse，可作为该 regime/win...
6,analysis.segment_effect_matrix.v1,window,stress_window,stress_window,,ridge_mse,2,ridge_mse,0.495000,gcn_mse,...,0.000000,gcn_mse,4.0000,ridge_mse,1.00000,3.000000e+00,gcn_mse,0.0,0.0,ridge_mse 在 stress_window 领先；gcn_mse 明显弱于基准，需检...
7,analysis.segment_effect_matrix.v1,window_regime,early::bull,early,bull,ridge_mse,2,gcn_mse,6.150000,ridge_mse,...,4.120000,gcn_mse,0.0000,gcn_mse,0.00000,0.000000e+00,gcn_mse,0.0,0.0,gcn_mse 在 early::bull 强于基准 ridge_mse，可作为该 regi...
8,analysis.segment_effect_matrix.v1,window_regime,early::neutral,early,neutral,ridge_mse,2,gcn_mse,0.000000,gcn_mse,...,0.000000,gcn_mse,0.0000,gcn_mse,0.00000,0.000000e+00,gcn_mse,0.0,0.0,early::neutral 模型差异较小，需结合回撤、换手和组合互补性判断。
9,analysis.segment_effect_matrix.v1,window_regime,stress_window::stress,stress_window,stress,ridge_mse,2,ridge_mse,0.495000,gcn_mse,...,0.000000,gcn_mse,4.0000,ridge_mse,1.00000,3.000000e+00,gcn_mse,0.0,0.0,ridge_mse 在 stress_window::stress 领先；gcn_mse 明...


## 中文结论

- 做了什么：新增并验证 `src.analysis.build_qlib_recorder_regime_window_analysis_report`，把多个 qlib recorder 的 `report_normal_1day` / `risk_analysis_1day` artifact 合并为统一 `AnalysisReport`。
- 核心指标：本实验生成 2 个模型、8 个交易日、16 行合并 portfolio report，并输出 `qlib_risk_metrics`、`segment_effect_matrix`、`model_effect_deltas`、`model_review_briefing` 等 sidecar。
- regime 差异：`gcn_mse` 在 bull/early 区间收益更强，但在 `stress_window::stress` 明显弱于 `ridge_mse`，review briefing 同时给出 `strength` 和 `weakness`。
- 是否晋升：不能直接晋升。该入口只证明 qlib recorder 到分析层的框架链路可用；真实晋升仍需要真实 SOXL/SOXX 数据、交易成本、压力窗口和 walk-forward 复核。
- 风险：当前 notebook 使用合成 recorder 数据，不代表真实收益；若真实 qlib artifact 缺少 regime 上下文，需要由数据/因子层提供 `regime_frame` 后再运行。
- 下一步：把真实 qlib recorder 输出接入该入口，作为 dashboard export 和 production notebook 的统一分析来源。